In [1]:
path = "pkl/June 27, 2024 Presidential Debate Transcript.pkl"

In [2]:
import pandas as pd
df = pd.read_pickle(path)

In [3]:
df = df.filter(items=['speaker', 'role', 'content', 'turn_number'])


In [4]:
# Initiate Neo4j instance

from neo4j import GraphDatabase

neo4j_uri = None
neo4j_user = None
neo4j_password = None

# CHANGE THIS if instance information changes 
neo4j_cred_path = "Neo4j Credentials.txt"

# Read the file and extract the credentials
with open("Neo4j Credentials.txt", 'r') as file:
    for line in file:
        if line.startswith("NEO4J_URI"):
            neo4j_uri = line.strip().split('=')[1]
        elif line.startswith("NEO4J_USERNAME"):
            neo4j_user = line.strip().split('=')[1]
        elif line.startswith("NEO4J_PASSWORD"):
            neo4j_password = line.strip().split('=')[1]


In [5]:
# Create a driver instance and session
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
session = driver.session()

In [6]:
from openai import OpenAI
import json
import os

# Initialize the OpenAI client
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key = api_key)

what should text property be?

In [7]:
# Text node
# property - name: name of file with extension and dir removed

def create_text_node(name):
    query = """
    CREATE (d:Text {name: $name})
    RETURN d
    """
    result = session.run(query, name=name)
    return result.single()

document_name = path.replace("pkl/", "").replace(".pkl", "")
result = create_text_node(document_name)


In [8]:
def is_valid_json(json_str):
    try:
        json_object = json.loads(json_str)
    except ValueError as e:
        return False, str(e)
    return True, json_object

def handle_response(response_text):
    # Remove any extraneous text or formatting
    response_text = response_text.strip()

    if response_text.startswith('```json'):
        response_text = response_text[7:]
    if response_text.endswith('```'):
        response_text = response_text[:-3]
        
    response_text = response_text.strip()
    # Validate JSON
    is_valid, result = is_valid_json(response_text)
    if is_valid:
        return result
    else:
        raise ValueError(f"Invalid JSON response: {result}")

In [11]:

# Knowledge node
# property - type: type of knowledge

def get_knowledge_type(text):

    assistant = """You are a knowledge classifier expert. Given a text, classify it as procedural,
    declarative, or conditional. Your results should be returned in an RFC-8259 compliant JSON of the
    following schema:
    
    {
        type: String,
        explanation: String
    }
    
    """

    prompt = f"{text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    return result


def create_knowledge_node(session, document_name, text):
    result = get_knowledge_type(text)

    type = result['type']
    explanation = result['explanation']
    
    query = """
    MATCH (d:Text {name: $document_name})
    CREATE (k:Knowledge {type: $type, explanation: $explanation})
    CREATE (d)-[:HAS]->(k)
    RETURN k
    """
    
    result = session.run(query, document_name=document_name, type=type, explanation=explanation)
    return result.single()


# Combine all the text together
combined_text = " ".join(df['content'].tolist())

with driver.session() as session:
     create_knowledge_node(session, document_name, combined_text)

In [13]:
# Document node
# property - turn_number, speaker, content, role

def create_document_node(tx, turn_number, speaker, content, role, text_node_name):
    query = """
    MATCH (t:Text {name: $text_node_name})
    CREATE (d:Document {
        turn_number: $turn_number,
        speaker: $speaker,
        content: $content,
        role: $role
    })
    CREATE (t)-[:HAS]->(d)
    """
    tx.run(query, turn_number=turn_number, speaker=speaker, content=content, role=role, text_node_name=text_node_name)


# Iterate over the rows and create nodes
with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(create_document_node, row['turn_number'], row['speaker'], row['content'], row['role'], document_name)


In [17]:
# Topic node
# property - name

list_of_topics = []

def get_topic(text):

    assistant = """You are a topic extraction expert. Given a piece of text and list of topics, parse the list of topics first.
    If an element in the list is relevant to the text, then include it in your response. If not, extract unique topics. 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        name: [String],
    }
    
    """

    prompt = f"Text: {text}, List of Topics: {', '.join(list_of_topics)}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)

    list_of_topics.extend(result["name"])
    
    return result


def create_topic_node(session, turn_number, content):

    topics = get_topic(content)["name"]
    
    for topic in topics:
        # Check if the topic already exists
        query = """
        MERGE (t:Topic {name: $topic_name})
        RETURN t
        """
        session.run(query, topic_name=topic)
        
        # Create the relationship Document -> HAS -> Topic
        query = """
        MATCH (d:Document {turn_number: $turn_number}), (t:Topic {name: $topic_name})
        MERGE (d)-[:HAS]->(t)
        """
        session.run(query, turn_number=turn_number, topic_name=topic)


with driver.session() as session:
    # Retrieve all Document nodes
    query = "MATCH (d:Document) RETURN d.turn_number AS turn_number, d.content AS content"
    documents = session.run(query)
    
    for record in documents:
        turn_number = record["turn_number"]
        content = record["content"]
        create_topic_node(session, turn_number, content)


In [18]:
# Clarity node
# property - readability_score, readability_expl, grammar_score, grammar_expl, score

def get_readability(text):

    assistant = """You are a readability scorer expert. Given a piece of text, evaluate its readability on a scale of [0.0, 1.0), with
    1.0 being readable by everyone and 0.0 being readable by no one.
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def get_grammar(text):

    assistant = """You are a grammar scorer expert. Given a piece of text, evaluate its use and efficacy of grammar on a scale of [0.0, 1.0), with
    1.0 being highly effective usage of of grammar and 0.0 being poor use of grammar with little efficacy.
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result


def create_clarity_node(session, turn_number, content):
    readability = get_readability(content)
    grammar = get_grammar(content)
    
    readability_score = readability['score']
    readability_expl = readability['explanation']
    grammar_score = grammar['score']
    grammar_expl = grammar['explanation']
    score = readability_score * grammar_score

    # Create Clarity node and establish relationship
    query = """
    MATCH (d:Document {turn_number: $turn_number})
    CREATE (c:Clarity {
        readability_score: $readability_score,
        readability_expl: $readability_expl,
        grammar_score: $grammar_score,
        grammar_expl: $grammar_expl,
        score: $score
    })
    CREATE (d)-[:HAS]->(c)
    RETURN c
    """
    session.run(query, turn_number=turn_number, readability_score=readability_score, readability_expl=readability_expl,
                grammar_score=grammar_score, grammar_expl=grammar_expl, score=score)
    


with driver.session() as session:
    # Retrieve all Document nodes
    query = "MATCH (d:Document) RETURN d.turn_number AS turn_number, d.content AS content"
    documents = session.run(query)
    
    for record in documents:
        turn_number = record["turn_number"]
        content = record["content"]
        create_clarity_node(session, turn_number, content)

In [21]:
# Humor node
# property - score, explanation

def get_humor(text):

    assistant = """You are a humor scorer expert. Given a piece of text, evaluate its use and efficacy of humor on a scale of [0.0, 1.0), with
    1.0 being highly effective usage of of humor and 0.0 being poor use of grammar with little humor.
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

# Function to create Humor node
def create_humor_node(session, turn_number, content):
    humor = get_humor(content)
    
    score = humor['score']
    explanation = humor['explanation']

    # Create Humor node and establish relationship
    query = """
    MATCH (d:Document {turn_number: $turn_number})
    CREATE (h:Humor {
        score: $score,
        explanation: $explanation
    })
    CREATE (d)-[:HAS]->(h)
    RETURN h
    """
    session.run(query, turn_number=turn_number, score=score, explanation=explanation)

with driver.session() as session:
        # Retrieve all Document nodes
        query = "MATCH (d:Document) RETURN d.turn_number AS turn_number, d.content AS content"
        documents = session.run(query)
        
        for record in documents:
            turn_number = record["turn_number"]
            content = record["content"]
            create_humor_node(session, turn_number, content)


In [23]:
# Cohesion node
# property - topic_relevancy_score, topic_relevancy_explanation, topic_continuity_score, topic_continuity_explanation, topic_transition_score, topic_transition_explanation, score


def get_topic_relevancy(comment, previous_comment1, previous_comment2, previous_comment3, previous_comment4, previous_comment5):

    assistant = """You are a topic relevancy expert. Given a piece of text and the previous comments, evaluate the relevancy of the current
    topic or subject matter to the immediate context or discussion in the range [0.0, 1.0).
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Comment: {comment}, Previous Comment 1: {previous_comment1}, Previous Comment 2: {previous_comment2}, Previous Comment 3: {previous_comment3}, Previous Comment 4: {previous_comment4}, Previous Comment 5: {previous_comment5}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def get_topic_continuity(comment):

    assistant = """You are a topic continuity expert. Given a piece of text, evaluate the smoothness and logical flow of topics
    or themes from one point to another in the text in the range [0.0, 1.0). 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Comment: {comment}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def get_topic_transition(comment):

    assistant = """You are a topic transition expert. Given a piece of text, evaluate the effectivness of transitions between different
    topics or ideas in the text in the range [0.0, 1.0). 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: String
    }
    
    """

    prompt = f"Comment: {comment}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def create_cohesion_node(session, document, previous_comments):
    turn_number = document['turn_number']
    content = document['content']

    if turn_number <= 5:
        topic_relevancy = {
            "score": 0.75,
            "explanation": "No previous 5 turns"
        }
    else:
        topic_relevancy = get_topic_relevancy(content, *previous_comments)

    topic_continuity = get_topic_continuity(content)
    topic_transition = get_topic_transition(content)

    topic_relevancy_score = topic_relevancy['score']
    topic_relevancy_expl = topic_relevancy['explanation']
    topic_continuity_score = topic_continuity['score']
    topic_continuity_expl = topic_continuity['explanation']
    topic_transition_score = topic_transition['score']
    topic_transition_expl = topic_transition['explanation']
    score = topic_relevancy_score * topic_continuity_score * topic_transition_score

    # Create Cohesion node and establish relationship
    query = """
    MATCH (d:Document {turn_number: $turn_number})
    CREATE (c:Cohesion {
        topic_relevancy_score: $topic_relevancy_score,
        topic_relevancy_explanation: $topic_relevancy_expl,
        topic_continuity_score: $topic_continuity_score,
        topic_continuity_explanation: $topic_continuity_expl,
        topic_transition_score: $topic_transition_score,
        topic_transition_explanation: $topic_transition_expl,
        score: $score
    })
    CREATE (d)-[:HAS]->(c)
    RETURN c
    """
    session.run(query, turn_number=turn_number, topic_relevancy_score=topic_relevancy_score, topic_relevancy_expl=topic_relevancy_expl,
                topic_continuity_score=topic_continuity_score, topic_continuity_expl=topic_continuity_expl,
                topic_transition_score=topic_transition_score, topic_transition_expl=topic_transition_expl, score=score)
    

with driver.session() as session:
        # Retrieve all Document nodes
        query = "MATCH (d:Document) RETURN d.turn_number AS turn_number, d.content AS content ORDER BY d.turn_number"
        documents = session.run(query)
        
        documents_list = list(documents)
        
        for i, document in enumerate(documents_list):
            previous_comments = [
                documents_list[i-1]["content"] if i-1 >= 0 else "",
                documents_list[i-2]["content"] if i-2 >= 0 else "",
                documents_list[i-3]["content"] if i-3 >= 0 else "",
                documents_list[i-4]["content"] if i-4 >= 0 else "",
                documents_list[i-5]["content"] if i-5 >= 0 else ""
            ]
            create_cohesion_node(session, document, previous_comments)


In [24]:
# Euphony node
# property -  three_syllable_count, three_syllable_words, short_vowels_count, short_vowels_words, stress_patterns_count, stress_pattern_words, count

def get_three_syl_count(text):

    assistant = """You are a syllable counter expert. Given a piece of text, count all the words with three or more syllables. 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        count: float,
        words: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result


def get_short_vowels(text):

    assistant = """You are a short vowels counter expert. Given a piece of text, count all the instances of a short vowel sound, which are
    vowels pronounced in a short and quick manner. 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        count: float,
        words: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def get_stress_patterns(text):

    assistant = """You are a stress pattern counter expert. Given a piece of text, count all the instances of words where the primary stress is 
    on the first syllable, affecting the rhythm and pronounciation of text. 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        count: float,
        words: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result

def create_euphony_node(session, turn_number, content):
    three_syl = get_three_syl_count(content)
    short_vowels = get_short_vowels(content)
    stress_patterns = get_stress_patterns(content)
    
    three_syllable_count = three_syl['count']
    three_syllable_words = three_syl['words']
    short_vowels_count = short_vowels['count']
    short_vowels_words = short_vowels['words']
    stress_patterns_count = stress_patterns['count']
    stress_pattern_words = stress_patterns['words']
    count = three_syllable_count + short_vowels_count + stress_patterns_count

    # Create Euphony node and establish relationship
    query = """
    MATCH (d:Document {turn_number: $turn_number})
    CREATE (e:Euphony {
        three_syllable_count: $three_syllable_count,
        three_syllable_words: $three_syllable_words,
        short_vowels_count: $short_vowels_count,
        short_vowels_words: $short_vowels_words,
        stress_patterns_count: $stress_patterns_count,
        stress_pattern_words: $stress_pattern_words,
        count: $count
    })
    CREATE (d)-[:HAS]->(e)
    RETURN e
    """
    session.run(query, turn_number=turn_number, three_syllable_count=three_syllable_count, three_syllable_words=three_syllable_words,
                short_vowels_count=short_vowels_count, short_vowels_words=short_vowels_words,
                stress_patterns_count=stress_patterns_count, stress_pattern_words=stress_pattern_words, count=count)
    

with driver.session() as session:
        # Retrieve all Document nodes
        query = "MATCH (d:Document) RETURN d.turn_number AS turn_number, d.content AS content"
        documents = session.run(query)
        
        for record in documents:
            turn_number = record["turn_number"]
            content = record["content"]
            create_euphony_node(session, turn_number, content)


In [ ]:
# Sound stylistics node
# property - 

def get_rhyme(text):

    assistant = """You are a rhyme expert. Given a piece of text, measure the use and pattern of rhyming words in the text where similar sounding words or endings
    are used to create musicality and cohesion in the range [0.0, 1.0). 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result


def get_alliteration(text):

    assistant = """You are an alliteration expert. Given a piece of text, measure the repitition of initial consonant sounds in closely positioned words
    to create a rhythmic and phonetic effect in the range [0.0, 1.0). 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result


def get_assonance(text):

    assistant = """You are an assonance expert. Given a piece of text, measure the repitition of vowel sounds within closely positioned words
    to create internal rhyming and musicality in the range [0.0, 1.0). 
    
    Your results should be returned in an RFC-8259 compliant JSON of the following schema:
    
    {
        score: float,
        explanation: [String]
    }
    
    """

    prompt = f"Text: {text}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": assistant},
            {"role": "user", "content": prompt},
        ]
    )

    result = response.choices[0].message.content
    result = handle_response(result)
    
    return result